## Local modules

The script imports these files, so they are written next to the notebook first.

In [ ]:
%%writefile utils.py
"""Shared helpers for the road damage detection scripts.

Not meant to be run directly. The other scripts in this folder import from it.
"""
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
DEFAULT_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)


# --------------------------------------------------------------------------- #
# General helpers
# --------------------------------------------------------------------------- #
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=float)


def resolve_weights(weights) -> str:
    """Use the path as given, or relative to the project root. Otherwise return it
    unchanged (so names like 'yolov8s.pt' can be auto-downloaded by Ultralytics)."""
    p = Path(weights)
    if p.exists():
        return str(p)
    if (PROJECT_ROOT / p).exists():
        return str(PROJECT_ROOT / p)
    return str(weights)


def _is_rtdetr(weights: str) -> bool:
    if "rtdetr" in weights.lower() or "rt-detr" in weights.lower():
        return True
    p = Path(weights)
    if p.suffix == ".pt" and p.exists():
        try:
            import torch

            ckpt = torch.load(str(p), map_location="cpu", weights_only=False)
            m = ckpt.get("ema") or ckpt.get("model")
            return type(m).__name__ == "RTDETRDetectionModel"
        except Exception:
            return False
    return False


def load_model(weights, task: str | None = "detect"):
    """Load a YOLO or RT-DETR model. RT-DETR needs its own class because it uses a
    different predictor, so it is detected from the file name or the checkpoint."""
    from ultralytics import RTDETR, YOLO

    weights = resolve_weights(weights)
    if _is_rtdetr(weights):
        return RTDETR(weights)
    return YOLO(weights, task=task) if task else YOLO(weights)


# --------------------------------------------------------------------------- #
# Dataset yaml helpers
# --------------------------------------------------------------------------- #
def resolve_data_yaml(data) -> Path:
    """Write a copy of the dataset yaml with an absolute `path:`.

    Ultralytics resolves relative dataset paths against its own datasets directory,
    not the current folder, so we always hand it an absolute path.
    """
    data_path = Path(data)
    if not data_path.exists():
        data_path = PROJECT_ROOT / data
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset yaml not found: {data}")
    with open(data_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root = Path(cfg.get("path", data_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()
    cfg["path"] = root.as_posix()
    out_dir = PROJECT_ROOT / "runs" / "_resolved"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / data_path.name
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out


def load_names(data_yaml) -> dict[int, str]:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg["names"]
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    return {int(k): v for k, v in names.items()}


def get_split_dirs(data_yaml, split: str) -> tuple[Path, Path]:
    """Return (images_dir, labels_dir) for a split of a resolved dataset yaml."""
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if split not in cfg:
        raise KeyError(f"Split '{split}' is not defined in {data_yaml}")
    rel = cfg[split]
    if isinstance(rel, list):
        rel = rel[0]
    img_dir = Path(cfg["path"]) / rel
    parts = list(img_dir.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    else:
        raise ValueError(f"Cannot derive the labels folder from {img_dir}")
    return img_dir, Path(*parts)


def list_images(folder) -> list[Path]:
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def read_yolo_labels(path) -> tuple[np.ndarray, np.ndarray]:
    """Read a YOLO label file. Returns (class_ids[n], boxes_xyxy_normalised[n, 4])."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    arr = np.loadtxt(path, ndmin=2)
    cls = arr[:, 0].astype(int)
    cx, cy, w, h = arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
    xyxy = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)
    return cls, xyxy


# --------------------------------------------------------------------------- #
# Detection metrics (F1 at IoU 0.5, the RDD challenge metric)
# --------------------------------------------------------------------------- #
def box_iou_np(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between boxes a[n, 4] and b[m, 4] in xyxy format."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-12)


class DetectionEvaluator:
    """Accumulates predictions over a dataset and computes P / R / F1 at any
    confidence threshold.

    Predictions are matched to ground truth once (greedy, by descending confidence,
    same class, IoU >= iou_thr). Because greedy matching only depends on higher
    scoring boxes, the result for any threshold is just a filter on that single
    matching, so sweeping thresholds is cheap.

    IoU is computed on normalised coordinates, which gives the same value as pixel
    coordinates (IoU is invariant to scaling each axis).
    """

    def __init__(self, nc: int, iou_thr: float = 0.5):
        self.nc = nc
        self.iou_thr = iou_thr
        self.n_gt = np.zeros(nc, dtype=int)
        self._scores, self._cls, self._tp = [], [], []
        self._cache = None

    def add(self, pred_boxes, pred_scores, pred_cls, gt_boxes, gt_cls) -> None:
        self._cache = None
        pred_boxes = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)
        pred_scores = np.asarray(pred_scores, dtype=float).reshape(-1)
        pred_cls = np.asarray(pred_cls).astype(int).reshape(-1)
        gt_boxes = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)
        gt_cls = np.asarray(gt_cls).astype(int).reshape(-1)

        for c in range(self.nc):
            g = gt_boxes[gt_cls == c]
            self.n_gt[c] += len(g)
            m = pred_cls == c
            if not m.any():
                continue
            b, s = pred_boxes[m], pred_scores[m]
            order = np.argsort(-s)
            b, s = b[order], s[order]
            tp = np.zeros(len(b), dtype=bool)
            if len(g):
                ious = box_iou_np(b, g)
                matched = np.zeros(len(g), dtype=bool)
                for i in range(len(b)):
                    cand = np.where(matched, -1.0, ious[i])
                    j = int(cand.argmax())
                    if cand[j] >= self.iou_thr:
                        matched[j] = True
                        tp[i] = True
                        if matched.all():
                            break
            self._scores.append(s)
            self._cls.append(np.full(len(b), c, dtype=int))
            self._tp.append(tp)

    def _arrays(self):
        if self._cache is None:
            if self._scores:
                self._cache = (
                    np.concatenate(self._scores),
                    np.concatenate(self._cls),
                    np.concatenate(self._tp),
                )
            else:
                self._cache = (np.zeros(0), np.zeros(0, dtype=int), np.zeros(0, dtype=bool))
        return self._cache

    @staticmethod
    def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f1

    def at_threshold(self, conf: float) -> dict:
        scores, cls, tp = self._arrays()
        keep = scores >= conf
        tp_c = np.bincount(cls[keep & tp], minlength=self.nc)
        fp_c = np.bincount(cls[keep & ~tp], minlength=self.nc)
        fn_c = self.n_gt - tp_c
        per_class = []
        for c in range(self.nc):
            p, r, f1 = self._prf(int(tp_c[c]), int(fp_c[c]), int(fn_c[c]))
            per_class.append(
                {"class_id": c, "precision": p, "recall": r, "f1": f1,
                 "tp": int(tp_c[c]), "fp": int(fp_c[c]), "fn": int(fn_c[c]),
                 "n_gt": int(self.n_gt[c])}
            )
        p, r, f1 = self._prf(int(tp_c.sum()), int(fp_c.sum()), int(fn_c.sum()))
        return {"conf": float(conf), "precision": p, "recall": r, "f1": f1,
                "tp": int(tp_c.sum()), "fp": int(fp_c.sum()), "fn": int(fn_c.sum()),
                "per_class": per_class}

    def average_precision(self) -> np.ndarray:
        """Per-class AP at the evaluator's IoU threshold (101-point interpolation, as in COCO)."""
        scores, cls, tp = self._arrays()
        ap = np.zeros(self.nc)
        recall_points = np.linspace(0, 1, 101)
        for c in range(self.nc):
            if self.n_gt[c] == 0:
                continue
            m = cls == c
            order = np.argsort(-scores[m], kind="stable")
            hits = tp[m][order]
            tpc = np.cumsum(hits)
            fpc = np.cumsum(~hits)
            recall = tpc / self.n_gt[c]
            precision = tpc / np.maximum(tpc + fpc, 1)
            # precision envelope, then sample it at fixed recall points
            envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
            idx = np.searchsorted(recall, recall_points, side="left")
            ap[c] = np.mean([envelope[i] if i < len(envelope) else 0.0 for i in idx])
        return ap

    def sweep(self, thresholds=DEFAULT_THRESHOLDS) -> list[dict]:
        return [self.at_threshold(t) for t in thresholds]

    def best(self, thresholds=DEFAULT_THRESHOLDS):
        """Returns (best_overall, best_per_class, all_rows)."""
        rows = self.sweep(thresholds)
        best = max(rows, key=lambda r: r["f1"])
        best_pc = []
        for c in range(self.nc):
            br = max(rows, key=lambda r: r["per_class"][c]["f1"])
            best_pc.append({"conf": br["conf"], **br["per_class"][c]})
        return best, best_pc, rows


# --------------------------------------------------------------------------- #
# Results logging
# --------------------------------------------------------------------------- #
def append_metrics_row(row: dict, path=None) -> Path:
    """Append one row to results/metrics.csv (created if missing)."""
    import pandas as pd

    path = Path(path) if path else PROJECT_ROOT / "results" / "metrics.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
    return path

# `make_split.py`

```text
Create leak-free train / val / test splits from the converted RDD2022 data.

Why not a random split? RDD images are frames captured along roads, and
consecutive frames look almost identical. A random split puts near-duplicates in
both train and val, which inflates the scores. Here, images are grouped into
blocks of consecutive file numbers per country (e.g. Japan_000100 ... Japan_000149)
and whole blocks are assigned to a single split. This is a heuristic, since the
dataset has no explicit sequence IDs: increase --block-size if you suspect
longer sequences.

Splits are stratified by country (each country contributes roughly the same
fractions). For a domain-shift study, use --holdout-countries to send entire
countries to the test split instead.

Run after convert_voc_to_yolo.py:

    python scripts/make_split.py --root data/yolo
    python scripts/make_split.py --root data/yolo --holdout-countries Norway   # cross-country test

Outputs under <root>:

    images/{train,val,test}/    labels/{train,val,test}/
    splits.csv                  image -> split assignment

It also writes configs/rdd2022.yaml (use --skip-yaml to leave it alone).
```

In [ ]:
import argparse
import os
import random
import re
import shutil
import sys
from pathlib import Path

import pandas as pd
import yaml
from tqdm import tqdm

from utils import CLASS_NAMES, PROJECT_ROOT

In [ ]:
SPLITS = ("train", "val", "test")

In [ ]:
def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", default="data/yolo", help="folder produced by convert_voc_to_yolo.py")
    p.add_argument("--val-frac", type=float, default=0.15, help="fraction of blocks per country for val")
    p.add_argument("--test-frac", type=float, default=0.10, help="fraction of blocks per country for test "
                                                                  "(ignored with --holdout-countries)")
    p.add_argument("--block-size", type=int, default=50, help="consecutive images treated as one sequence")
    p.add_argument("--holdout-countries", nargs="*", default=[], help="countries sent entirely to test")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--link", action="store_true", help="symlink images instead of copying (saves disk)")
    p.add_argument("--overwrite", action="store_true", help="replace existing split folders")
    p.add_argument("--yaml", default="configs/rdd2022.yaml", help="dataset yaml to write (relative to project root)")
    p.add_argument("--skip-yaml", action="store_true")
    return p.parse_args()

In [ ]:
def parse_idx(stem: str):
    m = re.search(r"(\d+)$", stem)
    return int(m.group(1)) if m else None

In [ ]:
def assign_splits(df: pd.DataFrame, args) -> pd.Series:
    rng = random.Random(args.seed)
    holdout = set(args.holdout_countries)
    test_frac = 0.0 if holdout else args.test_frac
    group_split = {}

    for country, sub in df.groupby("country"):
        groups = sorted(sub["group"].unique())
        if country in holdout:
            group_split.update({g: "test" for g in groups})
            continue
        rng.shuffle(groups)
        n = len(groups)
        n_test = int(round(n * test_frac))
        n_val = int(round(n * args.val_frac))
        if test_frac > 0 and n >= 3:
            n_test = max(n_test, 1)
        if args.val_frac > 0 and n >= 3:
            n_val = max(n_val, 1)
        for i, g in enumerate(groups):
            group_split[g] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")

    return df["group"].map(group_split)

In [ ]:
def place(src: Path, dst: Path, link: bool, state: dict) -> None:
    if link and not state["link_failed"]:
        try:
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            os.symlink(src.resolve(), dst)
            return
        except OSError:
            state["link_failed"] = True
            print("Symlinks are not available here (on Windows they need admin / developer mode). "
                  "Falling back to copying.")
    shutil.copy2(src, dst)

In [ ]:
def main():
    args = parse_args()
    if args.val_frac + args.test_frac >= 1:
        sys.exit("--val-frac + --test-frac must be below 1")
    root = Path(args.root)
    manifest = root / "manifest.csv"
    if not manifest.exists():
        sys.exit(f"{manifest} not found. Run convert_voc_to_yolo.py first.")

    df = pd.read_csv(manifest)
    unknown = set(args.holdout_countries) - set(df["country"].unique())
    if unknown:
        sys.exit(f"Unknown countries: {sorted(unknown)}. Available: {sorted(df['country'].unique())}")
    if args.holdout_countries:
        print(f"Holding out {args.holdout_countries} as the test split (--test-frac ignored).")

    idx = df["stem"].map(parse_idx)
    df["group"] = [f"{c}_{int(i) // args.block_size}" if pd.notna(i) else s
                   for c, i, s in zip(df["country"], idx, df["stem"])]
    df["split"] = assign_splits(df, args)

    # sanity check: no block may appear in two splits
    assert df.groupby("group")["split"].nunique().max() == 1, "a block leaked across splits"

    used = [s for s in SPLITS if (df["split"] == s).any()]
    for s in used:
        for kind in ("images", "labels"):
            d = root / kind / s
            if d.exists() and any(d.iterdir()):
                if not args.overwrite:
                    sys.exit(f"{d} already has files. Re-run with --overwrite to replace the splits.")
                shutil.rmtree(d)
            d.mkdir(parents=True, exist_ok=True)

    state = {"link_failed": False}
    label_src_dir = root / "all" / "labels"
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Placing files"):
        img_src = Path(row.image_path)
        place(img_src, root / "images" / row.split / f"{row.stem}{img_src.suffix.lower()}", args.link, state)
        shutil.copy2(label_src_dir / f"{row.stem}.txt", root / "labels" / row.split / f"{row.stem}.txt")

    df.drop(columns=["image_path", "label_path"]).to_csv(root / "splits.csv", index=False)

    # ---- summary ----
    cols = [f"n_{n}" for n in CLASS_NAMES.values()]
    summary = df.groupby("split").agg(images=("stem", "count"), empty=("n_boxes", lambda s: int((s == 0).sum())),
                                      boxes=("n_boxes", "sum"), **{c: (c, "sum") for c in cols})
    print("\nPer split:\n", summary.reindex(used).to_string())
    print("\nImages per country and split:\n", pd.crosstab(df["country"], df["split"]).reindex(columns=used).to_string())

    if not args.skip_yaml:
        try:
            path_value = root.resolve().relative_to(PROJECT_ROOT).as_posix()
        except ValueError:
            path_value = root.resolve().as_posix()
        cfg = {"path": path_value, "train": "images/train", "val": "images/val"}
        if "test" in used:
            cfg["test"] = "images/test"
        cfg["names"] = dict(CLASS_NAMES)
        yaml_path = PROJECT_ROOT / args.yaml
        yaml_path.parent.mkdir(parents=True, exist_ok=True)
        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print(f"\nWrote {yaml_path}")
    print("\nNext: python scripts/train.py --model yolov8s.pt --imgsz 640 --epochs 100")

## Arguments

Edit `ARGS` (the command-line arguments of the script), then run the next cell.

In [ ]:
import sys

ARGS = []
sys.argv = ['make_split.py'] + ARGS

In [ ]:
main()